# 02 · Limpieza

**Fase CRISP-DM:** *Data Preparation* (limpieza). Cada paso corrige un problema encontrado en `01_exploracion.ipynb`. El CSV original no se modifica: el resultado se guarda aparte.

- **Entrada:** `data/raw/10.000_Empresas_mas_Grandes_del_País.csv`
- **Salida:** `data/processed/empresas_limpias.parquet`

In [1]:
# Permite importar src/ desde la carpeta notebooks/
import sys
sys.path.append("..")

import pandas as pd

from src import utils

## 1. Carga, nombres de columnas y cifras

Columnas en snake_case, sin tildes y en inglés. Las cifras financieras llegan como texto (con signo de pesos, comas y espacios) y se convierten a número.

In [2]:
df = utils.load_raw()
df = utils.rename_columns(df)
df[utils.FINANCIAL_COLS] = df[utils.FINANCIAL_COLS].apply(utils.parse_amounts)

print("Valores que no se pudieron convertir:", df[utils.FINANCIAL_COLS].isna().sum().sum())
df[utils.FINANCIAL_COLS].head()

Valores que no se pudieron convertir: 0


,operating_revenue,net_income,total_assets,total_liabilities,total_equity
0,100.59,9.03,198.11,114.35,83.76
1,26.40,0.63,8.11,4.70,3.41
2,22.40,-0.82,34.95,11.57,23.38
3,21.61,0.42,6.94,6.72,0.21
4,20.29,4.88,71.08,36.11,34.97


## 2. NIT, CIIU y año de corte

In [3]:
# NIT como texto y sin comas de miles
df["nit"] = df["nit"].astype(str).str.replace(",", "", regex=False)

# CIIU (actividad económica) como texto de 4 dígitos: 610 -> "0610"
df["ciiu"] = (
    df["ciiu"]
    .astype(str)
    .str.replace(",", "", regex=False)
    # Quedarse con la parte antes del punto decimal
    .str.split(".").str[0]
    .str.strip()
    .str.zfill(4)
)

df["year"] = df["year"].astype(str).str.replace(",", "", regex=False).astype(int)

# 2025 se necesita para saber qué empresas tuvieron dificultades en 2025 (conjunto de prueba)
assert df["year"].max() >= 2025, "El CSV no incluye 2025: descargar la versión actual (ver README)"

df[["nit", "ciiu", "year"]].head()

,nit,ciiu,year
0,899999068,0610,2025
1,830095213,4661,2025
2,900112515,1921,2025
3,900276962,4711,2025
4,890904996,3513,2025


## 3. Ciudad de domicilio

Se unifican variantes del mismo municipio (tildes, sufijos, nombres largos).

In [4]:
city_map = {
    # Aglutinaciones y separación de palabras
    "DON MATIAS": "DONMATIAS",
    "DOS QUEBRADAS": "DOSQUEBRADAS",
    "CAMPO ALEGRE": "CAMPOALEGRE",
    # Sufijos departamentales y aclaraciones
    "MAICAO GUAJIRA": "MAICAO",
    "MITU VAUPES": "MITU",
    "PUERTO GAITAN (MANACACIAS)": "PUERTO GAITAN",
    "LA PAZ (ROBLES)": "LA PAZ",
    "SAN ANDRES Y PROVIDENCIA": "SAN ANDRES",
    # Nombres oficiales largos vs. comunes
    "GUADALAJARA DE BUGA": "BUGA",
    "VILLA DE SAN DIEGO DE UBATE": "UBATE",
    "VILLA DE SAN DIEGO DE VILLA DE SAN DIEGO DE UBATE": "UBATE",
    "VILLA ROSARIO": "VILLA DEL ROSARIO",
    # Variantes que aparecen en 2025
    "EL ESPINAL": "ESPINAL",
    "EL GUAMO": "GUAMO",
    "GIRALDOTA": "GIRARDOTA",
    # Textos corruptos o truncados
    "SAN SEBASTIAN DE BUENAVISMAGDALENA": "SAN SEBASTIAN DE BUENAVISTA",
}

df["city"] = utils.normalize_text(df["city"]).replace(city_map)

In [5]:
# Ferretería Central (Boyacá): el nombre del departamento quedó pegado al final de la razón social
is_ferreteria = df["company_name"].str.contains("FERRETERIA CENTRAL", na=False) & (
    df["department"] == "BOYACA"
)
df.loc[is_ferreteria, "city"] = "PUERTO BOYACA"
# Quitar "BOYACA" del final del nombre ($ = fin del texto) y los espacios sobrantes
df.loc[is_ferreteria, "company_name"] = (
    df.loc[is_ferreteria, "company_name"].str.replace(r"BOYACA$", "", regex=True).str.strip()
)

## 4. Departamento

Se unifican nombres, se reubican capitales que quedaron en la columna de departamento y se corrigen cruces ciudad-departamento.

In [6]:
df["department"] = utils.normalize_text(df["department"])

# Capitales registradas como departamento: la ciudad pasa a su columna
capitals_in_department = {
    "ARMENIA": "QUINDIO",
    "MONTERIA": "CORDOBA",
    "VALLEDUPAR": "CESAR",
}
is_capital = df["department"].isin(capitals_in_department)
df.loc[is_capital, "city"] = df.loc[is_capital, "department"]
df.loc[is_capital, "department"] = df.loc[is_capital, "department"].map(capitals_in_department)

# Departamentos con dos nombres
df["department"] = df["department"].replace({
    "VALLE": "VALLE DEL CAUCA",
    "GUAJIRA": "LA GUAJIRA",
})

In [7]:
# Ciudades con un departamento mal asignado en algunos registros
city_to_dept = {
    "MEDELLIN": "ANTIOQUIA",
    "BARRANQUILLA": "ATLANTICO",
    "BUCARAMANGA": "SANTANDER",
    "PALMIRA": "VALLE DEL CAUCA",
    "RIOHACHA": "LA GUAJIRA",
    "CUCUTA": "NORTE DE SANTANDER",
    "OCANA": "NORTE DE SANTANDER",
    "BUCARASICA": "NORTE DE SANTANDER",
    "FUNZA": "CUNDINAMARCA",
    "COTA": "CUNDINAMARCA",
    "MOSQUERA": "CUNDINAMARCA",
    "LA TEBAIDA": "QUINDIO",
    "ARJONA": "BOLIVAR",
}
is_known_city = df["city"].isin(city_to_dept)
df.loc[is_known_city, "department"] = df.loc[is_known_city, "city"].map(city_to_dept)

# Bogotá: siempre Bogotá D.C. en ambas columnas
is_bogota = df["city"].isin(["BOGOTA D.C.", "BOGOTA, D.C.", "BOGOTA"])
df.loc[is_bogota, ["city", "department"]] = "BOGOTA D.C."

# Anserma en el Valle es Ansermanuevo, y "Satuario" en Risaralda es un error de digitación
df.loc[(df["city"] == "ANSERMA") & (df["department"] == "VALLE DEL CAUCA"), "city"] = "ANSERMANUEVO"
df.loc[(df["city"] == "SATUARIO") & (df["department"] == "RISARALDA"), "city"] = "SANTUARIO"

In [8]:
# Nombres oficiales DIVIPOLA (municipios con prefijos o sufijos omitidos)
official_names = {
    "CALI": "SANTIAGO DE CALI",
    "CARTAGENA": "CARTAGENA DE INDIAS",
    "CUCUTA": "SAN JOSE DE CUCUTA",
    "BUGA": "GUADALAJARA DE BUGA",
    "UBATE": "VILLA DE SAN DIEGO DE UBATE",
    "CARMEN DE VIBORAL": "EL CARMEN DE VIBORAL",
    "TUMACO": "SAN ANDRES DE TUMACO",
    "SANTAFE DE ANTIOQUIA": "SANTA FE DE ANTIOQUIA",
    "MARIQUITA": "SAN SEBASTIAN DE MARIQUITA",
    "PIENDAMO": "PIENDAMO - TUNIA",
    "SINCE": "SAN LUIS DE SINCE",
    "TOLU": "SANTIAGO DE TOLU",
    "INIRIDA GUAINIA": "INIRIDA",
    "ANTIOQUIA": "SANTA FE DE ANTIOQUIA",
}
df["city"] = df["city"].replace(official_names)

# Nombres ambiguos que en Antioquia corresponden a otro municipio
antioquia_names = {
    "SAN PEDRO": "SAN PEDRO DE LOS MILAGROS",
    "BOLIVAR": "CIUDAD BOLIVAR",
    "SANTUARIO": "EL SANTUARIO",
}
in_antioquia = (df["department"] == "ANTIOQUIA") & df["city"].isin(antioquia_names)
df.loc[in_antioquia, "city"] = df.loc[in_antioquia, "city"].map(antioquia_names)

**Verificación.** Ya no deben quedar cruces ciudad-departamento imposibles. Las ciudades que siguen en más de un departamento deben ser homónimos legítimos.

In [9]:
impossible_pairs = {
    "BOGOTA D.C.": ["ANTIOQUIA", "CUNDINAMARCA"],
    "MEDELLIN": ["MAGDALENA"],
    "BARRANQUILLA": ["BOLIVAR", "SAN ANDRES Y PROVIDENCIA"],
    "CUCUTA": ["SANTANDER"],
}
is_impossible = pd.concat(
    [(df["city"] == city) & df["department"].isin(depts) for city, depts in impossible_pairs.items()],
    axis=1,
).any(axis=1)
print("Filas con cruces imposibles:", is_impossible.sum())

depts_per_city = df.groupby("city")["department"].nunique()
print("Ciudades en más de un departamento:", depts_per_city[depts_per_city > 1].index.tolist())

Filas con cruces imposibles: 0
Ciudades en más de un departamento: ['GRANADA', 'GUAMAL', 'LA UNION', 'LA VEGA', 'RESTREPO']


In [10]:
# Homónimos legítimos: municipios distintos con el mismo nombre
homonyms = ["EL SANTUARIO", "GRANADA", "GUAMAL", "LA UNION", "LA VEGA", "RESTREPO"]
(
    df[df["city"].isin(homonyms)][["city", "department"]]
    .drop_duplicates()
    .sort_values(["city", "department"])
    .reset_index(drop=True)
)

,city,department
0,EL SANTUARIO,ANTIOQUIA
1,GRANADA,CUNDINAMARCA
2,GRANADA,META
3,GUAMAL,MAGDALENA
4,GUAMAL,META
5,LA UNION,ANTIOQUIA
6,LA UNION,NARINO
7,LA UNION,VALLE DEL CAUCA
8,LA VEGA,CAUCA
9,LA VEGA,CUNDINAMARCA


In [11]:
departments = sorted(df["department"].unique())
print(f"Entidades territoriales: {len(departments)}")
print(departments)

Entidades territoriales: 33
['AMAZONAS', 'ANTIOQUIA', 'ARAUCA', 'ATLANTICO', 'BOGOTA D.C.', 'BOLIVAR', 'BOYACA', 'CALDAS', 'CAQUETA', 'CASANARE', 'CAUCA', 'CESAR', 'CHOCO', 'CORDOBA', 'CUNDINAMARCA', 'GUAINIA', 'GUAVIARE', 'HUILA', 'LA GUAJIRA', 'MAGDALENA', 'META', 'NARINO', 'NORTE DE SANTANDER', 'PUTUMAYO', 'QUINDIO', 'RISARALDA', 'SAN ANDRES Y PROVIDENCIA', 'SANTANDER', 'SUCRE', 'TOLIMA', 'VALLE DEL CAUCA', 'VAUPES', 'VICHADA']


## 5. Región

En 2025 cambian los nombres de las regiones y en 2021-2024 hay asignaciones inconsistentes. Se reconstruye la región desde el departamento, con la clasificación de 2025.

In [12]:
region_by_department = (
    df[df["year"] == 2025]
    .groupby("department")["region"]
    # La región más frecuente de cada departamento
    .agg(lambda s: s.mode()[0])
    .to_dict()
)
# Guainía no aparece en 2025, así que se asigna igual que Vaupés y Vichada
region_by_department["GUAINIA"] = "LLANOS"

df["region"] = df["department"].map(region_by_department)
print("Registros sin región:", df["region"].isna().sum())
df.groupby("region")["department"].unique()

Registros sin región: 0


region
ANTIOQUIA                                                      [ANTIOQUIA]
BOGOTÁ - CUNDINAMARCA                          [BOGOTA D.C., CUNDINAMARCA]
CENTRO - ORIENTE                   [SANTANDER, NORTE DE SANTANDER, BOYACA]
CENTRO - SUR                  [HUILA, TOLIMA, CAQUETA, PUTUMAYO, AMAZONAS]
COSTA PACÍFICA                     [VALLE DEL CAUCA, CAUCA, NARINO, CHOCO]
EJE CAFETERO                                  [RISARALDA, CALDAS, QUINDIO]
LLANOS                   [META, CASANARE, ARAUCA, GUAVIARE, VAUPES, VIC...
REGIÓN CARIBE            [BOLIVAR, ATLANTICO, MAGDALENA, SUCRE, CORDOBA...
Name: department, dtype: object

## 6. Razón social por NIT

Un NIT puede tener varios nombres en distintos años. Se conserva el más reciente.

In [13]:
# Quitar espacios en los bordes y reducir los repetidos a uno solo
df["company_name"] = df["company_name"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

# Nombre del año más reciente de cada NIT, asignado a todas sus filas
latest_name = df.sort_values(["nit", "year"]).groupby("nit")["company_name"].last()
df["company_name"] = df["nit"].map(latest_name)

print("NITs con más de un nombre:", (df.groupby("nit")["company_name"].nunique() > 1).sum())

NITs con más de un nombre: 0


## 7. Entidades de Supersalud

Supersalud desaparece del ranking en 2025 y sus entidades siguen marcos contables especiales, no comparables con el resto. Se excluyen.

In [14]:
is_supersalud = df["supervisor"] == "SUPERSALUD"
print("Registros de Supersalud por año:", df[is_supersalud].groupby("year").size().to_dict())

df = df[~is_supersalud].reset_index(drop=True)
print("Registros restantes:", len(df))

Registros de Supersalud por año: {2021: 921, 2022: 787, 2023: 710, 2024: 467}
Registros restantes: 47115


## 8. Resultado y exportación

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 47115 entries, 0 to 47114
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   nit                47115 non-null  str    
 1   company_name       47115 non-null  str    
 2   supervisor         47115 non-null  str    
 3   region             47115 non-null  str    
 4   department         47115 non-null  str    
 5   city               47115 non-null  str    
 6   ciiu               47115 non-null  object 
 7   macrosector        47115 non-null  str    
 8   operating_revenue  47115 non-null  float64
 9   net_income         47115 non-null  float64
 10  total_assets       47115 non-null  float64
 11  total_liabilities  47115 non-null  float64
 12  total_equity       47115 non-null  float64
 13  year               47115 non-null  int64  
dtypes: float64(5), int64(1), object(1), str(7)
memory usage: 9.5+ MB


In [16]:
utils.save(df, utils.CLEAN_PATH)
print("Guardado en", utils.CLEAN_PATH.relative_to(utils.ROOT))

Guardado en data\processed\empresas_limpias.parquet
